# 📱 Liquidity Stress Prediction Challenge - Starter Notebook

Welcome to the my notebook for this challenge.

In this challenge, my goal is to build a machine learning model that predicts whether a customer is likely to experience liquidity stress, using historical mobile money behaviour.

This notebook walks through a simple end-to-end pipeline:

- Loading and exploring the data
- Preprocessing
- Training a baseline model
- Evaluating performance
- Generating predictions for submission

The baseline model used here is **Logistic Regression**.


In [1]:
# 📚 Importing Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression


sns.set_style("whitegrid")
pd.set_option("display.max_columns", 200)


In [2]:
# 📥 Load the Data
train = pd.read_csv("Train.csv")
test = pd.read_csv("Test.csv")

# Peek at the data
train.head()


,arpu,age,gender,region,smartphone,segment,earning_pattern,x_90_d_activity_rate,m1_paybill_volume,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_volume,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_volume,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_volume,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_volume,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_volume,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_volume,m1_merchantpay_total_value,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_volume,m2_merchantpay_total_value,m2_merchantpay_highest_amount,m2_merchantpay_merchants,m3_merchantpay_volume,m3_merchantpay_total_value,m3_merchantpay_highest_amount,m3_merchantpay_merchants,m4_merchantpay_volume,m4_merchantpay_total_value,m4_merchantpay_highest_amount,m4_merchantpay_merchants,m5_merchantpay_volume,m5_merchantpay_total_value,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_volume,m6_merchantpay_total_value,m6_merchantpay_highest_amount,m6_merchantpay_merchants,m1_transfer_from_bank_volume,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_volume,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_highest_amount,m3_transfer_from_bank_banks,m4_transfer_from_bank_volume,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_volume,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_volume,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_highest_amount,m1_mm_send_recipients,m2_mm_send_volume,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_highest_amount,m3_mm_send_recipients,m4_mm_send_volume,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_volume,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,m6_mm_send_volume,m6_mm_send_total_value,m6_mm_send_highest_amount,m6_mm_send_recipients,m1_received_volume,m1_received_total_value,m1_received_highest_amount,m1_received_senders,m2_received_volume,m2_received_total_value,m2_received_highest_amount,m2_received_senders,m3_received_volume,m3_received_total_value,m3_received_highest_amount,m3_received_senders,m4_received_volume,m4_received_total_value,m4_received_highest_amount,m4_received_senders,m5_received_volume,m5_received_total_value,m5_received_highest_amount,m5_received_senders,m6_received_volume,m6_received_total_value,m6_received_highest_amount,m6_received_senders,m1_deposit_volume,m1_deposit_total_value,m1_deposit_highest_amount,m1_deposit_agents,m2_deposit_volume,m2_deposit_total_value,m2_deposit_highest_amount,m2_deposit_agents,m3_deposit_volume,m3_deposit_total_value,m3_deposit_highest_amount,m3_deposit_agents,m4_deposit_volume,m4_deposit_total_value,m4_deposit_highest_amount,m4_deposit_agents,m5_deposit_volume,m5_deposit_total_value,m5_deposit_highest_amount,m5_deposit_agents,m6_deposit_volume,m6_deposit_total_value,m6_deposit_highest_amount,m6_deposit_agents,m1_withdraw_volume,m1_withdraw_total_value,m1_withdraw_highest_amount,m1_withdraw_agents,m2_withdraw_volume,m2_withdraw_total_value,m2_withdraw_highest_amount,m2_withdraw_agents,m3_withdraw_volume,m3_withdraw_total_value,m3_withdraw_highest_amount,m3_withdraw_agents,m4_withdraw_volume,m4_withdraw_total_value,m4_withdraw_highest_amount,m4_withdraw_agents,m5_withdraw_volume,m5_withdraw_total_value,m5_withdraw_highe

## Data cleaning and exploration

Here we inspect the data for important information that could affect the training process and model building.

In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Columns: 184 entries, arpu to ID
dtypes: float64(92), int64(86), object(6)
memory usage: 56.2+ MB


In [4]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Columns: 183 entries, arpu to ID
dtypes: float64(92), int64(85), object(6)
memory usage: 41.9+ MB


In [5]:
# Check missing values
missing_summary = train.isnull().sum().sort_values(ascending=False)
# missing_summary[missing_summary > 0].head(20)
missing_summary ## There are no missing values

arpu                                    0
m6_received_highest_amount              0
m4_received_total_value                 0
m4_received_highest_amount              0
m4_received_senders                     0
                                       ..
m2_transfer_from_bank_banks             0
m3_transfer_from_bank_volume            0
m3_transfer_from_bank_total_value       0
m3_transfer_from_bank_highest_amount    0
ID                                      0
Length: 184, dtype: int64

In [6]:
missing_summary[missing_summary > 0]

Series([], dtype: int64)

There are no missing values.

## Feature Engineering
Here we will create new columns that could give more meaning to the features. For example means, volatility and trends can be gotten from the monthly data. This will create a clear understanding of the patterns or overall behaviours of the candidates transactions.

In [7]:
train.columns

Index(['arpu', 'age', 'gender', 'region', 'smartphone', 'segment',
       'earning_pattern', 'x_90_d_activity_rate', 'm1_paybill_volume',
       'm1_paybill_total_value',
       ...
       'm6_withdraw_highest_amount', 'm6_withdraw_agents', 'm1_daily_avg_bal',
       'm2_daily_avg_bal', 'm3_daily_avg_bal', 'm4_daily_avg_bal',
       'm5_daily_avg_bal', 'm6_daily_avg_bal', 'liquidity_stress_next_30d',
       'ID'],
      dtype='object', length=184)

In [8]:
## features of concern
features_to_engineer=[
    'paybill_volume','paybill_total_value','paybill_highest_amount','paybill_companies',
    'merchantpay_volume','merchantpay_total_value','merchantpay_highest_amount','merchantpay_highest_amount','merchantpay_merchants',
    'transfer_from_bank_volume','transfer_from_bank_total_value','transfer_from_bank_highest_amount','transfer_from_bank_banks',
    'mm_send_volume','mm_send_total_value','mm_send_highest_amount','mm_send_recipients',
    'received_volume','received_total_value','received_highest_amount','received_senders',
    'deposit_volume','deposit_total_value','deposit_highest_amount','deposit_agents',
    'withdraw_volume','withdraw_total_value','withdraw_highest_amount','withdraw_agents',   
]

feat=['m1_daily_avg_bal','m2_daily_avg_bal','m3_daily_avg_bal','m4_daily_avg_bal','m5_daily_avg_bal','m6_daily_avg_bal']

In [9]:
EPS = 1e-6
T = np.array([6, 5, 4, 3, 2, 1], dtype=float)   # M1 (recent) = 6 ... M6 (old) = 1
TC = T - T.mean()
DENOM = TC @ TC

def _slope(x):
    return (x - x.mean(axis=1, keepdims=True)) @ TC / DENOM

def add_trend_features(df):
    new = {}
    for f in features_to_engineer:
        cols = [f"m{i}_{f}" for i in range(1, 7)]
        v = df[cols].fillna(0)                      # NaN month = no activity
        mean = v.mean(axis=1)
        new[f"{f}_mean"] = mean
        new[f"{f}_std"] = v.std(axis=1)
        recent = v[cols[:3]].mean(axis=1)
        older = v[cols[3:]].mean(axis=1)
        new[f"{f}_ratio"] = (recent + EPS) / (older + EPS)

        if f.endswith(("_volume", "_total_value")):
            s = _slope(v.to_numpy(dtype=float))
            new[f"{f}_slope"] = s
            new[f"{f}_rel_slope"] = s / (mean.to_numpy() + EPS)

    bal = [f"m{i}_daily_avg_bal" for i in range(1, 7)]
    b = df[bal]
    bal_mean = b.mean(axis=1)
    new["bal_mean"] = bal_mean
    new["bal_std"] = b.std(axis=1)
    new["bal_ratio"] = (b[bal[:3]].mean(axis=1) + EPS) / (b[bal[3:]].mean(axis=1) + EPS)
    bs = _slope(b.to_numpy(dtype=float))
    new["bal_slope"] = bs
    new["bal_rel_slope"] = bs / (bal_mean.to_numpy() + EPS)

    return pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)

train_engineered = add_trend_features(train)
test_engineered = add_trend_features(test)

In [10]:
train_engineered.shape

(40000, 301)

In [11]:
train_engineered.head()

,arpu,age,gender,region,smartphone,segment,earning_pattern,x_90_d_activity_rate,m1_paybill_volume,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_volume,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_volume,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_volume,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_volume,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_volume,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_volume,m1_merchantpay_total_value,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_volume,m2_merchantpay_total_value,m2_merchantpay_highest_amount,m2_merchantpay_merchants,m3_merchantpay_volume,m3_merchantpay_total_value,m3_merchantpay_highest_amount,m3_merchantpay_merchants,m4_merchantpay_volume,m4_merchantpay_total_value,m4_merchantpay_highest_amount,m4_merchantpay_merchants,m5_merchantpay_volume,m5_merchantpay_total_value,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_volume,m6_merchantpay_total_value,m6_merchantpay_highest_amount,m6_merchantpay_merchants,m1_transfer_from_bank_volume,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_volume,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_highest_amount,m3_transfer_from_bank_banks,m4_transfer_from_bank_volume,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_volume,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_volume,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_highest_amount,m1_mm_send_recipients,m2_mm_send_volume,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_highest_amount,m3_mm_send_recipients,m4_mm_send_volume,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_volume,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,...,merchantpay_volume_std,merchantpay_volume_ratio,merchantpay_volume_slope,merchantpay_volume_rel_slope,merchantpay_total_value_mean,merchantpay_total_value_std,merchantpay_total_value_ratio,merchantpay_total_value_slope,merchantpay_total_value_rel_slope,merchantpay_highest_amount_mean,merchantpay_highest_amount_std,merchantpay_highest_amount_ratio,merchantpay_merchants_mean,merchantpay_merchants_std,merchantpay_merchants_ratio,transfer_from_bank_volume_mean,transfer_from_bank_volume_std,transfer_from_bank_volume_ratio,transfer_from_bank_volume_slope,transfer_from_bank_volume_rel_slope,transfer_from_bank_total_value_mean,transfer_from_bank_total_value_std,transfer_from_bank_total_value_ratio,transfer_from_bank_total_value_slope,transfer_from_bank_total_value_rel_slope,transfer_from_bank_highest_amount_mean,transfer_from_bank_highest_amount_std,transfer_from_bank_highest_amount_ratio,transfer_from_bank_banks_mean,transfer_from_bank_banks_std,transfer_from_bank_banks_ratio,mm_send_volume_mean,mm_send_volume_std,mm_send_volume_ratio,mm_send_volume_slope,mm_send_volume_rel_slope,mm_send_total_value_mean,mm_send_total_value_std,mm_send_total_value_ratio,mm_send_total_value_slope,mm_send_total_value_rel_slope,mm_send_highest_amount_mean,mm_send_highest_amount_std,mm_send_highest_amount_ratio,mm_send_recipients_mean,mm_send_recipients_std,mm_send_recipients_ratio,received_volume_mean,received_volume_std,received_volume_ratio,received_volume_slope,received_volume_rel_slope,received_total_value_mean,received_total_value_std,received_tot

### 🏷️ Encoding Categorical Variables

We use **one-hot encoding** to convert categorical columns into numeric format so they can be used by the model.


In [12]:
# Identify the ID column, target column and feature columns
TARGET = "liquidity_stress_next_30d"
ID_COL = "ID"
object_type=['str','object']

feature_cols = [c for c in train_engineered.columns if c not in [ID_COL, TARGET]]
cat_cols = [c for c in feature_cols if train_engineered[c].dtype in object_type]
num_cols = (
    train_engineered.select_dtypes(include=['number'])
    .drop(columns=[TARGET], errors='ignore')
    .columns.tolist()
)

cat_cols


['gender', 'region', 'smartphone', 'segment', 'earning_pattern']

In [13]:
## Inspect categorical columns for misspelling
for col in cat_cols:
  print(f"Unique values in {col}:", train[col].unique())

Unique values in gender: ['F' 'M']
Unique values in region: ['Central Zone' 'Northern Highlands' 'Lake Zone (Northwest)'
 'Coastal & Eastern Corridor' 'Southern Highlands' 'Islander'
 'Western Zone']
Unique values in smartphone: ['Yes' 'No']
Unique values in segment: ['MVC' 'LVC' 'HVC']
Unique values in earning_pattern: ['Monthly Earner' 'Weekly Earner' 'Irregular Earner']


In [14]:
for col in cat_cols:
  print(f"Unique values in {col}:", test[col].unique())

Unique values in gender: ['F' 'M']
Unique values in region: ['Central Zone' 'Northern Highlands' 'Lake Zone (Northwest)'
 'Coastal & Eastern Corridor' 'Southern Highlands' 'Islander'
 'Western Zone']
Unique values in smartphone: ['Yes' 'No']
Unique values in segment: ['MVC' 'LVC' 'HVC']
Unique values in earning_pattern: ['Monthly Earner' 'Weekly Earner' 'Irregular Earner']


In [15]:
## Split to Features and Target
X=train_engineered[feature_cols]
y=train_engineered[TARGET]

Split the training data to training and validation. We apply statify to ensure that equal proportion of the 1s and 0s are accounted for within each split

In [16]:
X_train,X_val,y_train,y_val=train_test_split(X,y,test_size=0.3,random_state=42,stratify=y)

In [17]:
y_train.value_counts()

liquidity_stress_next_30d
0    23800
1     4200
Name: count, dtype: int64

In [18]:
y_val.value_counts()

liquidity_stress_next_30d
0    10200
1     1800
Name: count, dtype: int64

In [19]:
X_train.shape,X_val.shape

((28000, 299), (12000, 299))

We will now apply One Hot Encoding for the the categorical columns in the X_train and X_val

In [20]:
# 1. Initialize the encoder (drop='first' mimics what you used before)
ohe = OneHotEncoder(drop='first', sparse_output=False)

# 2. Fit and transform ONLY on the categorical training columns
train_encoded_array = ohe.fit_transform(X_train[cat_cols])
# 3. Transform ONLY on the categorical validation columns
val_encoded_array = ohe.transform(X_val[cat_cols])

# 4. Get the new feature names generated by OHE
encoded_feature_names = ohe.get_feature_names_out(cat_cols)

# 4. Convert the NumPy array back into a pandas DataFrame
X_train_encoded = pd.DataFrame(
    train_encoded_array,
    columns=encoded_feature_names,
    index=X_train.index
)

X_val_encoded = pd.DataFrame(
    val_encoded_array,
    columns=encoded_feature_names,
    index=X_val.index
)

X_train_encoded.head()


,gender_M,region_Coastal & Eastern Corridor,region_Islander,region_Lake Zone (Northwest),region_Northern Highlands,region_Southern Highlands,region_Western Zone,smartphone_Yes,segment_LVC,segment_MVC,earning_pattern_Monthly Earner,earning_pattern_Weekly Earner
39167,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
17329,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
20411,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
19056,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1001,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0


In [21]:
# 3. Drop the old unencoded categorical columns from X_train and X_val
# and concatenate the new OHE columns alongside the numeric ones
X_train_ohe = pd.concat(
    [X_train.drop(columns=cat_cols), X_train_encoded],
    axis=1
)

X_val_ohe = pd.concat(
    [X_val.drop(columns=cat_cols), X_val_encoded],
    axis=1
)

X_train_ohe.head()

,arpu,age,x_90_d_activity_rate,m1_paybill_volume,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_volume,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_volume,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_volume,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_volume,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_volume,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_volume,m1_merchantpay_total_value,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_volume,m2_merchantpay_total_value,m2_merchantpay_highest_amount,m2_merchantpay_merchants,m3_merchantpay_volume,m3_merchantpay_total_value,m3_merchantpay_highest_amount,m3_merchantpay_merchants,m4_merchantpay_volume,m4_merchantpay_total_value,m4_merchantpay_highest_amount,m4_merchantpay_merchants,m5_merchantpay_volume,m5_merchantpay_total_value,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_volume,m6_merchantpay_total_value,m6_merchantpay_highest_amount,m6_merchantpay_merchants,m1_transfer_from_bank_volume,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_volume,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_highest_amount,m3_transfer_from_bank_banks,m4_transfer_from_bank_volume,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_volume,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_volume,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_highest_amount,m1_mm_send_recipients,m2_mm_send_volume,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_highest_amount,m3_mm_send_recipients,m4_mm_send_volume,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_volume,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,m6_mm_send_volume,m6_mm_send_total_value,m6_mm_send_highest_amount,m6_mm_send_recipients,m1_received_volume,...,merchantpay_merchants_mean,merchantpay_merchants_std,merchantpay_merchants_ratio,transfer_from_bank_volume_mean,transfer_from_bank_volume_std,transfer_from_bank_volume_ratio,transfer_from_bank_volume_slope,transfer_from_bank_volume_rel_slope,transfer_from_bank_total_value_mean,transfer_from_bank_total_value_std,transfer_from_bank_total_value_ratio,transfer_from_bank_total_value_slope,transfer_from_bank_total_value_rel_slope,transfer_from_bank_highest_amount_mean,transfer_from_bank_highest_amount_std,transfer_from_bank_highest_amount_ratio,transfer_from_bank_banks_mean,transfer_from_bank_banks_std,transfer_from_bank_banks_ratio,mm_send_volume_mean,mm_send_volume_std,mm_send_volume_ratio,mm_send_volume_slope,mm_send_volume_rel_slope,mm_send_total_value_mean,mm_send_total_value_std,mm_send_total_value_ratio,mm_send_total_value_slope,mm_send_total_value_rel_slope,mm_send_highest_amount_mean,mm_send_highest_amount_std,mm_send_highest_amount_ratio,mm_send_recipients_mean,mm_send_recipients_std,mm_send_recipients_ratio,received_volume_mean,received_volume_std,received_volume_ratio,received_volume_slope,received_volume_rel_slope,received_total_value_mean,received_total_value_std,received_total_value_ratio,received_total_value_slope,received_total_value_rel_slope,received_highest_amount_mean,received_highest_amount_std,received_highest_amount_ratio,received_senders_mean,received_senders_std,received_senders_ratio,deposit_volume_mean,deposit_volume_std,deposit_volume_ratio,depo

### Inspect for Feature Importance to the target variable using the correlation
We will apply correlation to see the linear relationship between each feature and the target

In [22]:
# 1. Calculate correlation with y_train and convert the resulting Series to a DataFrame
correlation_df = X_train_ohe.corrwith(y_train).to_frame(name='Correlation')

# 2. Sort by absolute correlation to see the most impactful features at the top
correlation_df = (
    correlation_df
    .assign(Abs_Correlation=correlation_df['Correlation'].abs())
    .sort_values(by='Abs_Correlation', ascending=False)
    .drop(columns=['Abs_Correlation'])
)

# Display the DataFrame
correlation_df

,Correlation
bal_rel_slope,-0.251231
withdraw_volume_ratio,0.224364
bal_ratio,-0.212569
withdraw_agents_ratio,0.189933
deposit_total_value_rel_slope,-0.178434
...,...
region_Lake Zone (Northwest),-0.001088
region_Northern Highlands,-0.000969
m1_mm_send_recipients,0.000900
m3_deposit_total_value,0.000895


The Linear relationships are very weak this could mean the relationships are non-linear

### CHECK FOR COLLINEARITY (within the features)
We will check for collinearity amongst the numerical variables.

In [23]:
## Correlation function that find the linear relationship between features using a given threshold to identify collinear relationship

def correlation(dataset, threshold):
    col_corr = set()
    col_pair = set()

    corr_matrix = dataset.corr()

    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if abs(corr_matrix.iloc[i, j]) > threshold:
                colname1 = corr_matrix.columns[i]
                colname2 = corr_matrix.columns[j]

                col_pair.add((colname1, colname2, corr_matrix.iloc[i, j]))
                col_corr.add(colname1)

    return col_corr, col_pair

In [24]:
col_corr, col_pair = correlation(X_train_ohe[num_cols], 0.7)
col_pair

{('transfer_from_bank_highest_amount_mean',
  'm5_deposit_highest_amount',
  0.7131340102476748),
 ('deposit_highest_amount_mean',
  'm6_merchantpay_highest_amount',
  0.7244475004027602),
 ('bal_mean', 'm4_daily_avg_bal', 0.8627015703013851),
 ('mm_send_highest_amount_std',
  'm1_withdraw_highest_amount',
  0.7588766126960121),
 ('mm_send_highest_amount_std',
  'mm_send_highest_amount_mean',
  0.8926518733688126),
 ('mm_send_highest_amount_std', 'm5_daily_avg_bal', 0.7775196436694147),
 ('paybill_highest_amount_std',
  'paybill_total_value_mean',
  0.7893162418952158),
 ('paybill_total_value_mean', 'm2_daily_avg_bal', 0.7098974291707608),
 ('transfer_from_bank_highest_amount_std',
  'm4_mm_send_highest_amount',
  0.772591556971141),
 ('deposit_highest_amount_std', 'm1_daily_avg_bal', 0.7842987751265327),
 ('bal_rel_slope', 'bal_ratio', 0.8265466033780379),
 ('m6_withdraw_highest_amount', 'arpu', 0.7224300130502139),
 ('mm_send_total_value_mean', 'mm_send_volume_mean', 0.86536391504001

These are the collinear pairs. We will have to drop some of the pairs systematically in a way that we compare with the relationship with the target. 

In [25]:
threshold = 0.7

# Strength of each feature's relationship with the target
target_corr = X_train_ohe[num_cols].corrwith(y_train).abs()

# Pairwise feature correlations (upper triangle only, so each pair appears once)
corr = X_train_ohe[num_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))

pairs = (
    upper.stack()
    .rename("pair_corr")
    .reset_index()
    .rename(columns={"level_0": "feat_a", "level_1": "feat_b"})
)
pairs = pairs[pairs["pair_corr"] > threshold].sort_values("pair_corr", ascending=False)

# For each pair, drop the feature with the weaker target correlation
to_drop, log = set(), []
for _, r in pairs.iterrows():
    a, b = r["feat_a"], r["feat_b"]
    if a in to_drop or b in to_drop:      # pair already resolved by an earlier drop
        continue
    drop, keep = (a, b) if target_corr[a] < target_corr[b] else (b, a)
    to_drop.add(drop)
    log.append({
        "dropped": drop,
        "kept": keep,
        "pair_corr": round(r["pair_corr"], 3),
        "dropped_target_corr": round(target_corr[drop], 4),
        "kept_target_corr": round(target_corr[keep], 4),
        "gap": round(target_corr[keep] - target_corr[drop], 4),
    })

drop_log = pd.DataFrame(log)

# Pairs where the two features are nearly tied on target correlation: review by hand
close_calls = drop_log[drop_log["gap"] < 0.01]

drop_log

,dropped,kept,pair_corr,dropped_target_corr,kept_target_corr,gap
0,transfer_from_bank_volume_rel_slope,transfer_from_bank_total_value_rel_slope,0.953,0.0543,0.0611,0.0068
1,merchantpay_volume_rel_slope,merchantpay_total_value_rel_slope,0.953,0.0303,0.0305,0.0002
2,transfer_from_bank_total_value_mean,transfer_from_bank_total_value_std,0.946,0.0170,0.0196,0.0025
3,bal_mean,bal_std,0.936,0.0441,0.0779,0.0338
4,deposit_total_value_mean,deposit_total_value_std,0.935,0.0228,0.0256,0.0028
...,...,...,...,...,...,...
118,m3_withdraw_highest_amount,m3_withdraw_volume,0.711,0.0315,0.0410,0.0095
119,m6_withdraw_highest_amount,m6_withdraw_volume,0.709,0.0087,0.0133,0.0046
120,m2_withdraw_highest_amount,m2_withdraw_volume,0.707,0.0215,0.0347,0.0132
121,m5_withdraw_volume,m5_withdraw_highest_amount,0.706,0.0082,0.0158,0.0076


In [26]:
## number of columns to drop
len(to_drop)

123

In [27]:
## convert to list
features_to_drop=list(to_drop)
features_to_drop

['m4_merchantpay_volume',
 'm5_merchantpay_total_value',
 'merchantpay_total_value_mean',
 'transfer_from_bank_banks_mean',
 'm6_merchantpay_volume',
 'm3_mm_send_total_value',
 'paybill_total_value_mean',
 'm6_paybill_total_value',
 'm4_withdraw_highest_amount',
 'm1_withdraw_total_value',
 'deposit_volume_ratio',
 'deposit_volume_rel_slope',
 'withdraw_highest_amount_std',
 'received_highest_amount_mean',
 'received_senders_ratio',
 'merchantpay_highest_amount_std',
 'merchantpay_volume_rel_slope',
 'm2_deposit_highest_amount',
 'merchantpay_merchants_std',
 'm2_withdraw_highest_amount',
 'mm_send_highest_amount_mean',
 'm3_mm_send_highest_amount',
 'm4_paybill_volume',
 'm3_deposit_highest_amount',
 'm5_mm_send_total_value',
 'm6_merchantpay_total_value',
 'm3_withdraw_highest_amount',
 'm3_received_highest_amount',
 'deposit_highest_amount_mean',
 'received_total_value_ratio',
 'mm_send_total_value_mean',
 'bal_std',
 'm5_deposit_volume',
 'm6_mm_send_highest_amount',
 'm3_merchant

## Deduction

Majority of the engineered columns are dropped when we used the pipeline we used for previous feature selection

In [28]:
features_to_drop_prev=[
 'm4_merchantpay_volume',
 'm2_deposit_highest_amount',
 'm2_merchantpay_volume',
 'm5_deposit_volume',
 'arpu',
 'm1_paybill_volume',
 'm3_paybill_volume',
 'm1_deposit_highest_amount',
 'm1_merchantpay_volume',
 'm6_merchantpay_total_value',
 'm6_merchantpay_volume',
 'm1_mm_send_highest_amount',
 'm5_paybill_volume',
 'm4_withdraw_highest_amount',
 'm1_received_highest_amount',
 'm4_transfer_from_bank_volume',
 'm6_withdraw_highest_amount',
 'm5_withdraw_volume',
 'm3_deposit_highest_amount',
 'm4_merchantpay_highest_amount',
 'm5_merchantpay_total_value',
 'm5_transfer_from_bank_volume',
 'm2_daily_avg_bal',
 'm4_deposit_volume',
 'm3_merchantpay_volume',
 'm1_merchantpay_total_value',
 'm2_withdraw_highest_amount',
 'm1_withdraw_highest_amount',
 'm6_mm_send_volume',
 'm6_deposit_volume',
 'm3_received_highest_amount',
 'm3_transfer_from_bank_highest_amount',
 'm2_paybill_volume',
 'm5_merchantpay_volume',
 'm2_merchantpay_highest_amount',
 'm4_received_volume',
 'm4_paybill_volume',
 'm3_merchantpay_highest_amount',
 'm2_transfer_from_bank_volume',
 'm2_received_highest_amount',
 'm1_transfer_from_bank_volume',
 'm5_received_volume',
 'm6_transfer_from_bank_volume',
 'm6_received_volume',
 'm3_mm_send_highest_amount',
 'm3_daily_avg_bal',
 'm2_mm_send_volume',
 'm5_mm_send_volume',
 'm3_withdraw_highest_amount',
 'm4_mm_send_volume',
 'm6_paybill_volume']

In [29]:
len(features_to_drop_prev)

51

In [30]:
X_train_ohe_adj=X_train_ohe.drop(columns=features_to_drop_prev)
X_train_ohe_adj.head()

,age,x_90_d_activity_rate,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_total_value,m2_merchantpay_merchants,m3_merchantpay_total_value,m3_merchantpay_merchants,m4_merchantpay_total_value,m4_merchantpay_merchants,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_highest_amount,m6_merchantpay_merchants,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_banks,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_recipients,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_recipients,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,m6_mm_send_total_value,m6_mm_send_highest_amount,m6_mm_send_recipients,m1_received_volume,m1_received_total_value,m1_received_senders,m2_received_volume,m2_received_total_value,m2_received_senders,m3_received_volume,m3_received_total_value,m3_received_senders,m4_received_total_value,m4_received_highest_amount,m4_received_senders,m5_received_total_value,m5_received_highest_amount,m5_received_senders,m6_received_total_value,m6_received_highest_amount,m6_received_senders,m1_deposit_volume,m1_deposit_total_value,m1_deposit_agents,m2_deposit_volume,m2_deposit_total_value,m2_deposit_agents,m3_deposit_volume,m3_deposit_total_value,m3_deposit_agents,m4_deposit_total_value,m4_deposit_highest_amount,m4_deposit_agents,m5_deposit_total_value,m5_deposit_highest_amount,...,merchantpay_merchants_mean,merchantpay_merchants_std,merchantpay_merchants_ratio,transfer_from_bank_volume_mean,transfer_from_bank_volume_std,transfer_from_bank_volume_ratio,transfer_from_bank_volume_slope,transfer_from_bank_volume_rel_slope,transfer_from_bank_total_value_mean,transfer_from_bank_total_value_std,transfer_from_bank_total_value_ratio,transfer_from_bank_total_value_slope,transfer_from_bank_total_value_rel_slope,transfer_from_bank_highest_amount_mean,transfer_from_bank_highest_amount_std,transfer_from_bank_highest_amount_ratio,transfer_from_bank_banks_mean,transfer_from_bank_banks_std,transfer_from_bank_banks_ratio,mm_send_volume_mean,mm_send_volume_std,mm_send_volume_ratio,mm_send_volume_slope,mm_send_volume_rel_slope,mm_send_total_value_mean,mm_send_total_value_std,mm_send_total_value_ratio,mm_send_total_value_slope,mm_send_total_value_rel_slope,mm_send_highest_amount_mean,mm_send_highest_amount_std,mm_send_highest_amount_ratio,mm_send_recipients_mean,mm_send_recipients_std,mm_send_recipients_ratio,received_volume_mean,received_volume_std,received_volume_ratio,received_volume_slope,received_volume_rel_slope,received_total_value_mean,received_total_value_std,received_total_value_ratio,received_total_value_slope,received_total_value_rel_slope,received_highest_amount_mean,received_highest_amount_std,received_highest_amount_ratio,received_senders_mean,received_senders_std,received_senders_ratio,deposit_volume_mean,deposit_volume_std,deposit_volume_ratio,deposit_volume_slope,deposit_volume_rel_slope,depo

In [31]:
X_val_ohe_adj=X_val_ohe.drop(columns=features_to_drop_prev)


In [32]:
X_val_ohe_adj.shape, X_train_ohe_adj.shape

((12000, 255), (28000, 255))

## Grid Search

We will apply grid search to identify the best parameters to get optimal result. 

In [33]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score,log_loss
from sklearn.linear_model import LogisticRegression


In [34]:
param_distributions = {
        "num_leaves": [15, 31],
        "learning_rate": [0.05, 0.1],
        "n_estimators": [200, 400],
        "class_weight": [None, "balanced"],
        "colsample_bytree": [0.8, 1.0],
        "max_depth": [-1, 6, 10],
        "subsample": [0.8, 1.0]
}

In [35]:
from lightgbm import LGBMClassifier

In [ ]:
## Run grid
RANDOM_STATE=42
model=LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)

## Scale the data
scaler=StandardScaler()
X_train_ohe_scaled=scaler.fit_transform(X_train_ohe)
X_val_ohe_scaled=scaler.transform(X_val_ohe)

random=RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions,
        n_iter=100,
        cv=3,
        verbose=2,
        n_jobs=-1
    )
random.fit(X_train_ohe_scaled,y_train) ## Note we are not using oversampled data
best_parameters=random.best_params_

print(f'______________ Best Params for Catboost Regression _________________')
print(best_parameters)

c:\Users\pc\Downloads\Financial Stress Prediction\venv\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 96 is smaller than n_iter=100. Running 96 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 96 candidates, totalling 288 fits
______________ Best Params for Catboost Regression _________________
{'random_strength': 5, 'learning_rate': 0.1, 'l2_leaf_reg': 5, 'iterations': 400, 'depth': 4, 'auto_class_weights': 'None'}


In [36]:
best_parameters={'subsample': 0.8, 'num_leaves': 31, 'n_estimators': 400, 'max_depth': -1, 'learning_rate': 0.05, 'colsample_bytree': 0.8, 'class_weight': None}

# best_parameters={'random_strength': 1, 'learning_rate': 0.1, 'l2_leaf_reg': 5, 'iterations': 400, 'depth': 6, 'auto_class_weights': 'Balanced'}

In [37]:
from sklearn.base import clone
import pandas as pd
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             classification_report, confusion_matrix)


In [38]:
RANDOM_STATE=42
model=LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)


In [39]:
## Scale the data
scaler=StandardScaler()
X_train_ohe_adj_scaled=scaler.fit_transform(X_train_ohe_adj)
X_val_ohe_adj_scaled=scaler.transform(X_val_ohe_adj)

In [56]:
print(f"LightGBM training")

# fresh copy of the estimator with the tuned params applied
model_obj = clone(model).set_params(**best_parameters)
model_obj.fit(X_train_ohe_adj_scaled, y_train)

# probabilities and labels
train_prob = model_obj.predict_proba(X_train_ohe_adj_scaled)[:, 1]
val_prob = model_obj.predict_proba(X_val_ohe_adj_scaled)[:, 1]
val_pred = (val_prob >= 0.5).astype(int)
train_pred = (train_prob >= 0.5).astype(int)

train_auc = roc_auc_score(y_train, train_prob)
val_auc = roc_auc_score(y_val, val_prob)
rep = classification_report(y_val, val_pred, output_dict=True)

print(f"Train ROC-AUC: {train_auc} | Validation ROC-AUC: {val_auc}")
print(f"Train accuracy: {accuracy_score(y_train, train_pred)} | "
        f"Validation accuracy: {accuracy_score(y_val, val_pred)}")
## Log-loss for the predicted classes
print(f"Train prediction log_loss: {log_loss(y_train, train_pred)} | "
        f"Validation prediction log_loss: {log_loss(y_val, val_pred)}")
## Log-loss for the predicted probability
print(f"Train probability log loss: {log_loss(y_train, train_prob)} | Val probability log loss: {log_loss(y_val, val_prob)}")

print(classification_report(y_val, val_pred))
print(confusion_matrix(y_val, val_pred))
print("\n" + "-" * 60 + "\n")


LightGBM training
Train ROC-AUC: 0.9966433673469387 | Validation ROC-AUC: 0.9013978758169935
Train accuracy: 0.9735357142857143 | Validation accuracy: 0.8959166666666667
Train prediction log_loss: 0.9538695414762791 | Validation prediction log_loss: 3.7515435902506105
Train probability log loss: 0.11007396871042678 | Val probability log loss: 0.2565786139175384
              precision    recall  f1-score   support

           0       0.91      0.97      0.94     10200
           1       0.75      0.46      0.57      1800

    accuracy                           0.90     12000
   macro avg       0.83      0.72      0.75     12000
weighted avg       0.89      0.90      0.88     12000

[[9929  271]
 [ 978  822]]

------------------------------------------------------------



The prediction of the minority class is low especially the recall. Let us inspect the performance with a better threshold. The log-loss for predicted value is based on the natural threshold selected at 0.5. Our focus should be on the log-loss of the prredicted probability which is 
`Train probability log loss: 0.11007396871042678 | Val probability log loss: 0.2565786139175384`

In [57]:
from sklearn.metrics import precision_recall_curve

In [58]:
# 7. Evaluate with best threshold

# 8. Tune threshold on validation (F1 for class 1)
prec, rec, thr = precision_recall_curve(y_val, val_prob)
f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
best_thr = thr[np.argmax(f1)]
print("Best threshold:", round(float(best_thr), 3))

val_pred = (val_prob >= best_thr).astype(int)

print("performance and classification_report with best threshold")
print(classification_report(y_val, val_pred))
print(confusion_matrix(y_val, val_pred))

Best threshold: 0.319
performance and classification_report with best threshold
              precision    recall  f1-score   support

           0       0.93      0.93      0.93     10200
           1       0.62      0.62      0.62      1800

    accuracy                           0.89     12000
   macro avg       0.78      0.78      0.78     12000
weighted avg       0.89      0.89      0.89     12000

[[9520  680]
 [ 685 1115]]


Improved recall.

## Create submission 

In [43]:
SUBMISSION_TARGET = "Target"

# Create a sample submission template directly from the test IDs
submission = pd.DataFrame({
    "ID": test["ID"],
    SUBMISSION_TARGET: 0.5
})


In [44]:
test_clean=test_engineered.drop(columns=['ID'])

In [45]:
# Apply one hot encoding
# 2. Fit transform ONLY on the categorical training columns
test_encoded_array = ohe.transform(test_clean[cat_cols])

# 3. Get the new feature names generated by OHE
encoded_feature_names = ohe.get_feature_names_out(cat_cols)

# 4. Convert the NumPy array back into a pandas DataFrame
test_encoded = pd.DataFrame(
    test_encoded_array,
    columns=encoded_feature_names,
    index=test_clean.index
)

test_encoded.head()

,gender_M,region_Coastal & Eastern Corridor,region_Islander,region_Lake Zone (Northwest),region_Northern Highlands,region_Southern Highlands,region_Western Zone,smartphone_Yes,segment_LVC,segment_MVC,earning_pattern_Monthly Earner,earning_pattern_Weekly Earner
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0


In [46]:
test_ohe = pd.concat(
    [test_clean.drop(columns=cat_cols), test_encoded],
    axis=1
)

test_ohe.head()

,arpu,age,x_90_d_activity_rate,m1_paybill_volume,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_volume,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_volume,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_volume,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_volume,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_volume,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_volume,m1_merchantpay_total_value,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_volume,m2_merchantpay_total_value,m2_merchantpay_highest_amount,m2_merchantpay_merchants,m3_merchantpay_volume,m3_merchantpay_total_value,m3_merchantpay_highest_amount,m3_merchantpay_merchants,m4_merchantpay_volume,m4_merchantpay_total_value,m4_merchantpay_highest_amount,m4_merchantpay_merchants,m5_merchantpay_volume,m5_merchantpay_total_value,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_volume,m6_merchantpay_total_value,m6_merchantpay_highest_amount,m6_merchantpay_merchants,m1_transfer_from_bank_volume,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_volume,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_highest_amount,m3_transfer_from_bank_banks,m4_transfer_from_bank_volume,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_volume,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_volume,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_highest_amount,m1_mm_send_recipients,m2_mm_send_volume,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_highest_amount,m3_mm_send_recipients,m4_mm_send_volume,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_volume,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,m6_mm_send_volume,m6_mm_send_total_value,m6_mm_send_highest_amount,m6_mm_send_recipients,m1_received_volume,...,merchantpay_merchants_mean,merchantpay_merchants_std,merchantpay_merchants_ratio,transfer_from_bank_volume_mean,transfer_from_bank_volume_std,transfer_from_bank_volume_ratio,transfer_from_bank_volume_slope,transfer_from_bank_volume_rel_slope,transfer_from_bank_total_value_mean,transfer_from_bank_total_value_std,transfer_from_bank_total_value_ratio,transfer_from_bank_total_value_slope,transfer_from_bank_total_value_rel_slope,transfer_from_bank_highest_amount_mean,transfer_from_bank_highest_amount_std,transfer_from_bank_highest_amount_ratio,transfer_from_bank_banks_mean,transfer_from_bank_banks_std,transfer_from_bank_banks_ratio,mm_send_volume_mean,mm_send_volume_std,mm_send_volume_ratio,mm_send_volume_slope,mm_send_volume_rel_slope,mm_send_total_value_mean,mm_send_total_value_std,mm_send_total_value_ratio,mm_send_total_value_slope,mm_send_total_value_rel_slope,mm_send_highest_amount_mean,mm_send_highest_amount_std,mm_send_highest_amount_ratio,mm_send_recipients_mean,mm_send_recipients_std,mm_send_recipients_ratio,received_volume_mean,received_volume_std,received_volume_ratio,received_volume_slope,received_volume_rel_slope,received_total_value_mean,received_total_value_std,received_total_value_ratio,received_total_value_slope,received_total_value_rel_slope,received_highest_amount_mean,received_highest_amount_std,received_highest_amount_ratio,received_senders_mean,received_senders_std,received_senders_ratio,deposit_volume_mean,deposit_volume_std,deposit_volume_ratio,depo

In [47]:
# Adjust the number of features
test_ohe_adj=test_ohe.drop(columns=features_to_drop_prev)
test_ohe_adj.head()

,age,x_90_d_activity_rate,m1_paybill_total_value,m1_paybill_highest_amount,m1_paybill_companies,m2_paybill_total_value,m2_paybill_highest_amount,m2_paybill_companies,m3_paybill_total_value,m3_paybill_highest_amount,m3_paybill_companies,m4_paybill_total_value,m4_paybill_highest_amount,m4_paybill_companies,m5_paybill_total_value,m5_paybill_highest_amount,m5_paybill_companies,m6_paybill_total_value,m6_paybill_highest_amount,m6_paybill_companies,m1_merchantpay_highest_amount,m1_merchantpay_merchants,m2_merchantpay_total_value,m2_merchantpay_merchants,m3_merchantpay_total_value,m3_merchantpay_merchants,m4_merchantpay_total_value,m4_merchantpay_merchants,m5_merchantpay_highest_amount,m5_merchantpay_merchants,m6_merchantpay_highest_amount,m6_merchantpay_merchants,m1_transfer_from_bank_total_value,m1_transfer_from_bank_highest_amount,m1_transfer_from_bank_banks,m2_transfer_from_bank_total_value,m2_transfer_from_bank_highest_amount,m2_transfer_from_bank_banks,m3_transfer_from_bank_volume,m3_transfer_from_bank_total_value,m3_transfer_from_bank_banks,m4_transfer_from_bank_total_value,m4_transfer_from_bank_highest_amount,m4_transfer_from_bank_banks,m5_transfer_from_bank_total_value,m5_transfer_from_bank_highest_amount,m5_transfer_from_bank_banks,m6_transfer_from_bank_total_value,m6_transfer_from_bank_highest_amount,m6_transfer_from_bank_banks,m1_mm_send_volume,m1_mm_send_total_value,m1_mm_send_recipients,m2_mm_send_total_value,m2_mm_send_highest_amount,m2_mm_send_recipients,m3_mm_send_volume,m3_mm_send_total_value,m3_mm_send_recipients,m4_mm_send_total_value,m4_mm_send_highest_amount,m4_mm_send_recipients,m5_mm_send_total_value,m5_mm_send_highest_amount,m5_mm_send_recipients,m6_mm_send_total_value,m6_mm_send_highest_amount,m6_mm_send_recipients,m1_received_volume,m1_received_total_value,m1_received_senders,m2_received_volume,m2_received_total_value,m2_received_senders,m3_received_volume,m3_received_total_value,m3_received_senders,m4_received_total_value,m4_received_highest_amount,m4_received_senders,m5_received_total_value,m5_received_highest_amount,m5_received_senders,m6_received_total_value,m6_received_highest_amount,m6_received_senders,m1_deposit_volume,m1_deposit_total_value,m1_deposit_agents,m2_deposit_volume,m2_deposit_total_value,m2_deposit_agents,m3_deposit_volume,m3_deposit_total_value,m3_deposit_agents,m4_deposit_total_value,m4_deposit_highest_amount,m4_deposit_agents,m5_deposit_total_value,m5_deposit_highest_amount,...,merchantpay_merchants_mean,merchantpay_merchants_std,merchantpay_merchants_ratio,transfer_from_bank_volume_mean,transfer_from_bank_volume_std,transfer_from_bank_volume_ratio,transfer_from_bank_volume_slope,transfer_from_bank_volume_rel_slope,transfer_from_bank_total_value_mean,transfer_from_bank_total_value_std,transfer_from_bank_total_value_ratio,transfer_from_bank_total_value_slope,transfer_from_bank_total_value_rel_slope,transfer_from_bank_highest_amount_mean,transfer_from_bank_highest_amount_std,transfer_from_bank_highest_amount_ratio,transfer_from_bank_banks_mean,transfer_from_bank_banks_std,transfer_from_bank_banks_ratio,mm_send_volume_mean,mm_send_volume_std,mm_send_volume_ratio,mm_send_volume_slope,mm_send_volume_rel_slope,mm_send_total_value_mean,mm_send_total_value_std,mm_send_total_value_ratio,mm_send_total_value_slope,mm_send_total_value_rel_slope,mm_send_highest_amount_mean,mm_send_highest_amount_std,mm_send_highest_amount_ratio,mm_send_recipients_mean,mm_send_recipients_std,mm_send_recipients_ratio,received_volume_mean,received_volume_std,received_volume_ratio,received_volume_slope,received_volume_rel_slope,received_total_value_mean,received_total_value_std,received_total_value_ratio,received_total_value_slope,received_total_value_rel_slope,received_highest_amount_mean,received_highest_amount_std,received_highest_amount_ratio,received_senders_mean,received_senders_std,received_senders_ratio,deposit_volume_mean,deposit_volume_std,deposit_volume_ratio,deposit_volume_slope,deposit_volume_rel_slope,depo

In [48]:
## Scale the data
test_ohe_adj_scaled=scaler.transform(test_ohe_adj)
test_ohe_adj_scaled

array([[ 0.1846283 ,  0.81121636, -0.2674084 , ...,  0.97697919,
         1.74601919, -1.22739155],
       [ 0.1846283 ,  0.81121636,  0.4371055 , ...,  0.97697919,
         1.74601919, -1.22739155],
       [ 0.1846283 ,  0.81121636, -0.2674084 , ...,  0.97697919,
         1.74601919, -1.22739155],
       ...,
       [ 0.83737936,  1.21950823, -0.06120475, ...,  0.97697919,
        -0.57273139,  0.81473593],
       [ 0.83737936,  1.21950823, -0.01956956, ...,  0.97697919,
        -0.57273139,  0.81473593],
       [ 0.83737936,  1.21950823,  0.20413674, ...,  0.97697919,
        -0.57273139,  0.81473593]])

In [49]:
test_prob=model_obj.predict_proba(test_ohe_adj_scaled)[:, 1]

In [50]:
submission[SUBMISSION_TARGET] = test_prob

In [51]:
submission

,ID,Target
0,ID_ED91AC7A28,0.011432
1,ID_29975A9C7E,0.037497
2,ID_F59CE25650,0.006395
3,ID_E875F68D1A,0.002952
4,ID_2902B1C0D0,0.002354
...,...,...
29995,ID_3C7CF79AEE,0.006792
29996,ID_F1651E73C4,0.010874
29997,ID_FD3F0934CA,0.138308
29998,ID_DA6916D6C7,0.008075


just a quick inspection

In [52]:
classification = (test_prob >= 0.5).astype(int)

In [53]:
classification

array([0, 0, 0, ..., 0, 0, 0])

In [54]:
from collections import Counter
class_freq=Counter(classification)
class_freq

Counter({0: 27073, 1: 2927})

In [55]:
# Save to CSV
submission.to_csv("submission_LightGBM_feature_engineered_and_selection.csv", index=False)
print("Submission file saved!")


Submission file saved!


### Heavy Feature Reduction

We will identify the features that best affect the model and train based on the 